In [1]:
import cupynumeric as np
from src.simulate_quantization import matmul_mixed, quantize_any, dequantize_any, NATIVE

[0 - 1fbfb1f00]    0.000000 {5}{module_config}: Module numa can not detect resources.


In [2]:
A = np.array([[1.2, -3.4],[5.6, 0.0]], dtype=np.float32)
B = np.array([[0.5, 2.0],[-1.0, 0.25]], dtype=np.float32)
# native x tiny
C1 = matmul_mixed(A.astype(np.float64), "fp64", B, "fp8_e4m3")
# complex native x native
Ac = (A + 0.2j*A).astype(np.complex64)
Bc = (B - 0.1j*B).astype(np.complex64)
C2 = matmul_mixed(Ac, "fp16", Bc, "fp32");

In [3]:
Ac @ Bc, C2

(array([[ 4.0800004+0.4j  ,  1.5810001+0.155j],
        [ 2.856    +0.28j , 11.424    +1.12j ]], dtype=complex64),
 array([[ 4.080486 +0.40021974j,  1.581279 +0.15494508j],
        [ 2.8567734+0.28004885j, 11.4270935+1.1201954j ]], dtype=complex64))

In [4]:
# --- map native format/dtype to appropriate target (real or complex) ---
def _choose_target_dtype(fmt_or_dtype, is_complex):
    # string formats "fp16/32/64"
    if isinstance(fmt_or_dtype, str) and fmt_or_dtype in NATIVE:
        base = NATIVE[fmt_or_dtype]
    elif hasattr(fmt_or_dtype, "kind"):  # actual dtype object
        base = fmt_or_dtype
    else:
        return None  # not a native fmt; tiny-fp path will handle

    if is_complex:
        # promote real floats to complex counterparts
        if base == np.float64:
            return np.complex128
        # there is no complex16 in NumPy; use complex64 for fp16/fp32
        return np.complex64
    else:
        # if caller already passed a complex dtype for a real array, keep it real
        if base in (np.complex64, np.complex128):
            return np.float64 if base == np.complex128 else np.float32
        return base

# ---------------- Casting helper (accepts dtype OR format str) ----------------
def cast_matrix_and_vector_mixed(H, psi, fmt_or_dtype, compute_dtype=np.float16):
    """
    If fmt_or_dtype is a numpy/cupynumeric dtype or 'fp16/fp32/fp64' -> cast (respecting complexness).
    If it's a tiny-fp format string -> emulate quant/dequant, then cast to compute_dtype.
    """
    tgt = _choose_target_dtype(fmt_or_dtype, np.iscomplexobj(psi) or np.iscomplexobj(H))
    if tgt is not None:
        # Preserve Hermiticity: cast H and ψ consistently
        return H.astype(tgt, copy=False), psi.astype(tgt, copy=False)

    # ---- tiny format path ----
    fmt = str(fmt_or_dtype)
    H_q = quantize_any(H, fmt)
    psi_q = quantize_any(psi, fmt)
    H_d = dequantize_any(H_q, fmt, compute_dtype)
    psi_d = dequantize_any(psi_q, fmt, compute_dtype)
    return H_d, psi_d

# ---------------- exp(-i H dt) • v using Hermitian eigendecomp ----------------
def expmv_hermitian(H, v, dt, work_dtype=np.float64, out_dtype=np.complex128, normalize=False):
    """
    Stable expmv for Hermitian H: use eigh. If H is complex, solve in complex dtype.
    """
    H_is_cplx = np.iscomplexobj(H)
    # Ensure appropriate working dtype
    if H_is_cplx and work_dtype in (np.float32, np.float64):
        work_dtype = np.complex128
    if not H_is_cplx and work_dtype in (np.complex64, np.complex128):
        work_dtype = np.float64

    Hh = H.astype(work_dtype, copy=False)
    vh = v.astype(out_dtype, copy=False)

    w, Q = np.linalg.eigh(Hh)  # H = Q diag(w) Q^H
    phase = np.exp((-1j * dt) * w.astype(out_dtype))
    y = Q.conj().T @ vh
    y *= phase
    out = (Q @ y).astype(out_dtype, copy=False)

    if normalize:
        n = np.linalg.norm(out)
        if n != 0:
            out /= n
    return out

# ---------------- Lie–Trotter with rotating mixed precision ----------------
def lie_trotter_step(H_terms, psi, dt, fmt_list, step_idx,
                     compute_dtype=np.float16, work_dtype=np.float32, out_dtype=np.complex64,
                     verbose=True, normalize=False):
    """
    Apply exp(-i H_j dt) in sequence. Each term uses a format selected from fmt_list,
    rotating per (step_idx + j). Each format can be:
      - native dtype object (e.g., np.float16/32/64) or "fp16"/"fp32"/"fp64"
      - tiny emulated string: "fp4_e2m1", "fp8_e4m3", "fp8_e5m2", etc.
    """
    psi_new = psi
    if verbose:
        sel = fmt_list[(step_idx) % len(fmt_list)]
        name = sel.name if hasattr(sel, "name") else str(sel)
        print(f"Step {step_idx}, format start: {name}")

    for j, H in enumerate(H_terms):
        fmt = fmt_list[(step_idx + j) % len(fmt_list)]
        Hc, psic = cast_matrix_and_vector_mixed(H, psi_new, fmt, compute_dtype=compute_dtype)
        psi_new = expmv_hermitian(Hc, psic, dt, work_dtype=work_dtype, out_dtype=out_dtype, normalize=normalize)
    return psi_new

def simulate_hamiltonian(H_terms, psi0, t, r, fmt_list,
                         compute_dtype=np.float16, work_dtype=np.float32, out_dtype=np.complex64,
                         verbose=True, normalize=False):
    """
    First-order Lie–Trotter: (∏_j e^{-i H_j Δt})^r ψ0
    """
    dt = t / r
    psi = psi0
    for step in range(r):
        psi = lie_trotter_step(
            H_terms, psi, dt, fmt_list, step,
            compute_dtype=compute_dtype, work_dtype=work_dtype, out_dtype=out_dtype,
            verbose=verbose, normalize=normalize
        )
    return psi

In [5]:
n = 8
# generate random Hermitian components H1 and H2
A = np.random.randn(n, n) + 1j * np.random.randn(n, n)
B = np.random.randn(n, n) + 1j * np.random.randn(n, n)
H1 = (A + A.conj().T) / 2
H2 = (B + B.conj().T) / 2
H_terms = [H1, H2]

# initial state |ψ₀⟩ random normalized vector
psi = np.random.randn(n) + 1j * np.random.randn(n)
psi /= np.linalg.norm(psi)

r = 10
t = 0.5
# Rotate through mixed precisions: tiny + native
# fmts = ["fp4_e2m1", "fp8_e4m3", "fp64", "fp8_e5m2"]
fmts = ["fp16", "fp32", "fp8_e4m3"]
# fmts = ["fp64"] * len(H_terms)  # all native fp64 for reference

psi_final = simulate_hamiltonian(H_terms, psi, t, r, fmts, verbose=True, normalize=True)

# normalize final state
psi_final /= np.linalg.norm(psi_final)

Step 0, format start: fp16
Step 1, format start: fp32
Step 2, format start: fp8_e4m3
Step 3, format start: fp16
Step 4, format start: fp32
Step 5, format start: fp8_e4m3
Step 6, format start: fp16
Step 7, format start: fp32
Step 8, format start: fp8_e4m3
Step 9, format start: fp16


In [6]:
psi_exact = np.linalg.expm(-1j * (H1 + H2) * t).dot(psi)

In [7]:
np.linalg.norm(psi_exact - psi_final)

array(0.09746797)

In [8]:
psi_exact, psi_final

(array([-0.1538611 -0.33513674j,  0.16657762-0.10627829j,
         0.26019737+0.03234759j, -0.48369865+0.09968353j,
         0.13858223-0.60368606j, -0.1528876 +0.10038094j,
         0.17082707-0.02502167j,  0.03512585+0.25334183j]),
 array([-0.14478774-0.34815004j,  0.19585888-0.12187947j,
         0.2754819 +0.04401983j, -0.53502923+0.08735391j,
         0.14929141-0.55187285j, -0.14844379+0.0957967j ,
         0.14628582-0.020713j  ,  0.05917445+0.2224937j ], dtype=complex64))

In [9]:
np.linalg.norm(psi_final)

array(1., dtype=float32)

In [14]:
n = 10
# generate random Hermitian components H1 and H2
A = np.random.randn(n, n) + 1j * np.random.randn(n, n)
B = np.random.randn(n, n) + 1j * np.random.randn(n, n)
H1 = (A + A.conj().T) / 2
H2 = (B + B.conj().T) / 2
H_terms = [H1, H2]

psi = np.random.randn(n) + 1j * np.random.randn(n)
psi /= np.linalg.norm(psi)


In [31]:
import cupynumeric as np

...

t = 0.4
r = 6

fmts = ["fp16", "fp32", "fp8_e4m3"]

psi_final = simulate_hamiltonian(H_terms, psi, t, r, fmts, 
                                 verbose=True, normalize=True)
psi_exact = np.linalg.expm(-1j * (H1 + H2) * t).dot(psi)
print("Approx error:", np.linalg.norm(psi_exact - psi_final))


Step 0, format start: fp16
Step 1, format start: fp32
Step 2, format start: fp8_e4m3
Step 3, format start: fp16
Step 4, format start: fp32
Step 5, format start: fp8_e4m3
Approx error: 0.13443868930500086
